# King County Housing — Exploratory Data Analysis

## Client

**Charles Christensen**, a seller. He wants big returns and is asking three
questions: should he renovate, which neighborhood, and when to sell.

## Data

21,597 sales of 21,420 houses in King County, WA, sold between May 2014 and
May 2015.

The data was pulled from the `eda` schema of the course database and joined
on the fly:

```sql
SELECT d.*, s.date, s.price
FROM eda.king_county_house_details AS d
INNER JOIN eda.king_county_house_sales AS s
        ON d.id = s.house_id
ORDER BY d.id, s.date;
```

The result was exported to `data/king_county_joined.csv`, which is not
tracked by git. Re-run the query above to reproduce it.

## Assumptions

- **Neighborhood** is operationalised as `zipcode`, the only geographic unit
  available in the data.
- **Return** is measured as price per square foot of living space, not
  absolute price, so that house size does not drive the result.
- **Repeat sales are real.** 176 houses were sold more than once. The
  shortest gap between two sales is 61 days and no sale date is duplicated,
  so these are genuine resales, not data-entry errors. All are kept.
- **Timing means seasonality.** The data spans a single year, so any timing
  effect is a within-year seasonal pattern, not a multi-year trend.

In [14]:
import warnings

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

plt.rcParams.update(
    {"figure.figsize": (8, 5), "axes.facecolor": "white", "axes.edgecolor": "black"}
)
plt.rcParams["figure.facecolor"] = "w"
pd.plotting.register_matplotlib_converters()
pd.set_option("display.float_format", lambda x: "%.3f" % x)

In [2]:
df = pd.read_csv("data/king_county_joined.csv")
df.shape

(21597, 21)

In [3]:
df.head()

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,price
0,1000102,6.000,3.000,2400.000,9373.000,2.000,NaN,0.000,3,7,...,0.000,1991,0.000,98002,47.326,-122.214,2060.000,7316.000,2014-09-16,280000.000
1,1000102,6.000,3.000,2400.000,9373.000,2.000,NaN,0.000,3,7,...,0.000,1991,0.000,98002,47.326,-122.214,2060.000,7316.000,2015-04-22,300000.000
2,1200019,4.000,1.750,2060.000,26036.000,1.000,NaN,0.000,4,8,...,900.000,1947,0.000,98166,47.444,-122.351,2590.000,21891.000,2014-05-08,647500.000
3,1200021,3.000,1.000,1460.000,43000.000,1.000,0.000,0.000,3,7,...,0.000,1952,0.000,98166,47.443,-122.347,2250.000,20023.000,2014-08-11,400000.000
4,2800031,3.000,1.000,1430.000,7599.000,1.500,0.000,0.000,4,6,...,420.000,1930,0.000,98168,47.478,-122.265,1290.000,10320.000,2015-04-01,235000.000


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21597 entries, 0 to 21596
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   id             21597 non-null  int64  
 1   bedrooms       21597 non-null  float64
 2   bathrooms      21597 non-null  float64
 3   sqft_living    21597 non-null  float64
 4   sqft_lot       21597 non-null  float64
 5   floors         21597 non-null  float64
 6   waterfront     19206 non-null  float64
 7   view           21534 non-null  float64
 8   condition      21597 non-null  int64  
 9   grade          21597 non-null  int64  
 10  sqft_above     21597 non-null  float64
 11  sqft_basement  21145 non-null  float64
 12  yr_built       21597 non-null  int64  
 13  yr_renovated   17749 non-null  float64
 14  zipcode        21597 non-null  int64  
 15  lat            21597 non-null  float64
 16  long           21597 non-null  float64
 17  sqft_living15  21597 non-null  float64
 18  sqft_lot15     21

In [5]:
df.describe()

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,price
count,21597.000,21597.000,21597.000,21597.000,21597.000,21597.000,19206.000,21534.000,21597.000,21597.000,21597.000,21145.000,21597.000,17749.000,21597.000,21597.000,21597.000,21597.000,21597.000,21597.000
mean,4580474287.771,3.373,2.116,2080.322,15099.409,1.494,0.008,0.234,3.410,7.658,1788.597,291.857,1971.000,836.651,98077.952,47.560,-122.214,1986.620,12758.284,540296.574
std,2876735715.748,0.926,0.769,918.106,41412.637,0.540,0.087,0.766,0.651,1.173,827.760,442.491,29.375,4000.111,53.513,0.139,0.141,685.230,27274.442,367368.140
min,1000102.000,1.000,0.500,370.000,520.000,1.000,0.000,0.000,1.000,3.000,370.000,0.000,1900.000,0.000,98001.000,47.156,-122.519,399.000,651.000,78000.000
25%,2123049175.000,3.000,1.750,1430.000,5040.000,1.000,0.000,0.000,3.000,7.000,1190.000,0.000,1951.000,0.000,98033.000,47.471,-122.328,1490.000,5100.000,322000.000
50%,3904930410.000,3.000,2.250,1910.000,7618.000,1.500,0.000,0.000,3.000,7.000,1560.000,0.000,1975.000,0.000,98065.000,47.572,-122.231,1840.000,7620.000,450000.000
75%,7308900490.000,4.000,2.500,2550.000,10685.000,2.000,0.000,0.000,4.000,8.000,2210.000,560.000,1997.000,0.000,98118.000,47.678,-122.125,2360.000,10083.000,645000.000
max,9900000190.000,33.000,8.000,13540.000,1651359.000,3.500,1.000,4.000,5.000,13.000,9410.000,4820.000,2015.000,20150.000,98199.000,47.778,-121.315,6210.000,871200.000,7700000.000


### First look — observations

**Shape.** 21,597 rows × 21 columns.

**Missing values.** Four columns are incomplete:

| Column | Non-null | Missing |
| --- | --- | --- |
| `waterfront` | 19,206 | 2,391 |
| `view` | 21,534 | 63 |
| `sqft_basement` | 21,145 | 452 |
| `yr_renovated` | 17,749 | 3,848 |

Every other column is complete.

**`date` is a string, not a date.** It has to be converted before any month or
season can be extracted.

**A 33-bedroom house.** `bedrooms` has a mean of 3.37 and a 75th percentile of
4, but a maximum of 33. Either a genuine mansion or a typo for 3 — needs to be
checked against its `sqft_living`.

**Heavily right-skewed columns.** `sqft_lot` has a median of 7,618 and a
maximum of 1,651,359, roughly 200× the typical lot. `price` shows the same
pattern: mean 540,297 against a median of 450,000. Wherever the mean sits well
above the median, a few very large values are stretching the right tail, so
this analysis reports **medians**, not means.

**`yr_renovated` mixes two things.** Its mean is 836.65, which is not a year at
all — it comes from averaging a column that is mostly 0 with a minority of
four-digit years. The 75th percentile is 0, so more than three quarters of the
houses were never renovated. The column is really a flag plus a date, and will
be split into both.

**Waterfront is rare.** The mean of `waterfront` is 0.008, i.e. under 1% of
sales, roughly 150 houses. Too small a group to carry the neighborhood
hypothesis on its own, so `zipcode` and coordinates will do most of that work.

In [6]:
df[df["bedrooms"] > 10]

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,price
4297,1773100755,11.000,3.000,3000.000,4960.000,2.000,0.000,0.000,3,7,...,600.000,1918,19990.000,98106,47.556,-122.363,1420.000,4960.000,2014-08-21,520000.000
6071,2402100895,33.000,1.750,1620.000,6000.000,1.000,0.000,0.000,5,7,...,580.000,1947,0.000,98103,47.688,-122.331,1330.000,4700.000,2014-06-25,640000.000


In [7]:
df[df["yr_renovated"] > 2015]

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,price
5,3600057,4.000,2.000,1650.000,3504.000,1.000,0.000,0.000,3,7,...,890.000,1951,20130.000,98144,47.580,-122.294,1480.000,3504.000,2015-03-19,402500.000
18,9000025,2.000,1.000,1420.000,4635.000,2.000,0.000,0.000,4,7,...,0.000,1941,19730.000,98115,47.680,-122.304,1810.000,4635.000,2014-12-03,496000.000
50,31000165,5.000,3.500,3620.000,7821.000,2.000,0.000,2.000,3,10,...,830.000,1958,20100.000,98040,47.574,-122.215,2690.000,9757.000,2014-09-11,1490000.000
72,46100204,5.000,3.000,3300.000,33474.000,1.000,NaN,3.000,3,9,...,1430.000,1957,19910.000,98040,47.567,-122.210,3836.000,20953.000,2015-02-21,1510000.000
74,46100504,4.000,3.750,4100.000,22798.000,1.500,NaN,3.000,5,11,...,1560.000,1934,19790.000,98040,47.565,-122.210,3880.000,18730.000,2014-06-17,2030000.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21462,9826701345,3.000,2.500,1620.000,2640.000,2.000,0.000,0.000,4,7,...,0.000,1900,19930.000,98122,47.604,-122.305,1370.000,3840.000,2014-07-15,498000.000
21476,9828200746,2.000,1.500,1120.000,1024.000,2.000,0.000,0.000,3,8,...,0.000,1970,19980.000,98122,47.617,-122.298,1120.000,1549.000,2015-05-04,440000.000
21492,9828700200,4.000,3.000,2170.000,4000.000,2.000,0.000,0.000,4,9,...,560.000,1982,20110.000,98112,47.620,-122.292,1670.000,4000.000,2014-05-05,831000.000
21505,9828701690,3.000,2.000,1530.000,3400.000,1.000,0.000,0.000,3,7,...,540.000,1907,20140.000,98112,47.620,-122.296,1880.000,4212.000,2014-08-06,529000.000


In [8]:
df[(df["yr_built"] < 1900) | (df["yr_built"] > 2015)]

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,price


In [9]:
df[(df["yr_renovated"] > 0) & (df["yr_renovated"] <= 2015)]

,id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,date,price


In [10]:
renovated = df[df["yr_renovated"] > 2015]
(renovated["yr_renovated"] % 10 == 0).sum()

np.int64(744)

### Data quality checks

**A 33-bedroom house is not trustworthy.** One house lists 33 bedrooms in
1,620 sqft on a single floor with 1.75 bathrooms — about 49 sqft per bedroom,
which is not a house. A typo for 3 is plausible, but with a single row and no
corroborating pattern there is no evidence for any particular correction, so
`bedrooms` is set to missing for this row rather than guessed. The rest of the
row (price, date, zipcode, size) is sound and is kept. `bedrooms` is not used
in any of the three hypotheses, so this choice does not affect the results.

**An 11-bedroom house is real.** 3,000 sqft over two floors with 3 bathrooms,
built in 1918 — roughly 272 sqft per room. Tight but plausible for a
century-old rooming house. Kept as a genuine outlier.

**`yr_renovated` is systematically multiplied by 10.** 744 rows hold five-digit
values such as 20130 or 19730. Two checks confirm this is systematic, not a
handful of typos:

- No row has a `yr_renovated` between 1 and 2015, so there are no valid
  four-digit values anywhere in the column.
- All 744 non-zero values are divisible by 10.

The whole column is therefore the real year × 10, and is divided by 10 during
cleaning. This leaves 744 renovated houses, about 4% of the houses whose
renovation status is known — a small but workable comparison group for
hypothesis 3.

**`yr_built` is clean.** No row falls outside 1900–2015.

### Research questions and hypotheses

| Question | Hypothesis | Indicators |
| --- | --- | --- |
| Does the time of year affect the sale price? | Houses sold in spring sell for more than houses sold in winter. | month of `date`, median `price` |
| Which neighborhoods deliver the highest returns? | Neighborhoods near the water and near the city centre have a higher price per square foot than outlying neighborhoods. | `zipcode`, `lat`, `long`, `price` / `sqft_living` |
| Does renovating before selling pay off? | Homes renovated recently sell for more than comparable non-renovated homes, while homes renovated long ago show no such premium. | `yr_renovated`, controlled for `grade` and `sqft_living` |

## Cleaning — step A: proven errors

In [18]:
df_clean = df.copy()

# date arrives from the CSV as a string
df_clean["date"] = pd.to_datetime(df_clean["date"])

# yr_renovated is systematically stored as the real year x 10
df_clean.loc[df_clean["yr_renovated"] > 0, "yr_renovated"] = (
    df_clean["yr_renovated"] / 10
)

# 33 bedrooms in 1620 sqft is not credible, and no correction is provable
df_clean.loc[df_clean["bedrooms"] == 33, "bedrooms"] = np.nan

# these four are conceptually whole numbers; they only became floats because
# a NumPy integer column cannot hold missing values. pandas' nullable Int64 can.
for col in ["yr_renovated", "bedrooms", "waterfront", "view"]:
    df_clean[col] = df_clean[col].astype("Int64")

In [19]:
print("date dtype      :", df_clean["date"].dtype)
print("yr_renovated max:", df_clean["yr_renovated"].max())
print("bedrooms max    :", df_clean["bedrooms"].max())
print("waterfront max  :", df_clean["waterfront"].max())
print("view max        :", df_clean["view"].max())
print("rows            :", len(df_clean), "of", len(df))

date dtype      : datetime64[us]
yr_renovated max: 2015
bedrooms max    : 11
waterfront max  : 1
view max        : 4
rows            : 21597 of 21597
